In [1]:
"""
HP Landslide Model — Fixed Training Script  (v6)
=================================================
Changes from v5:
  FIX A: Added CalibratedClassifierCV → probabilities now reflect real-world likelihood
  FIX B: Added more negative samples from dataset (reduces HIGH-bias)
  FIX C: Sanity checks now test the correct soil_moisture unit (0–25 range)
  FIX D: Thresholds raised: HIGH≥0.70  MEDIUM≥0.40
  FIX E: Saves scaler stats alongside model so Flask can document the units
  NOTE: elevation must be passed from district lookup — not from UI slider
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV   # FIX A
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, accuracy_score,
                              roc_auc_score, confusion_matrix)
import joblib

# ═══════════════════════════════════════════════
# 1. LOAD & CLEAN
# ═══════════════════════════════════════════════
df = pd.read_csv("C:/Users/babur/OneDrive/Desktop/ML_floodPrediction/data/processed/hp_landslide_dataset_v7_final.csv")

df['soil_moisture'] = df['soil_moisture'].fillna(df['soil_moisture'].median())
df['ndvi']          = df['ndvi'].clip(-1, 1)

# FIX B: Drop obvious outlier rows (ndvi > 2 is physically impossible)
df = df[df['ndvi'] <= 1.0]

print("=== Dataset shape:", df.shape)
print("\nLabel counts:")
print(df['label'].value_counts())
print(f"\nClass balance: {df['label'].mean():.1%} positive")

# ─── Correlation guard ────────────────────────────────────────────────
print("\n=== Correlation with label:")
corr = df.corr(numeric_only=True)['label'].sort_values(ascending=False)
print(corr.round(3))

rainfall_corr = abs(corr['rainfall_7day'])
assert rainfall_corr > 0.10, (
    f"❌ STOP: rainfall_7day correlation = {rainfall_corr:.3f}. Dataset problem."
)
print(f"\n✅ rainfall_7day correlation = {rainfall_corr:.3f}\n")

print("\nFeature ranges (DOCUMENT THESE — Flask must match units!):")
FEATURE_COLS = ['rainfall_7day', 'slope', 'elevation', 'soil_moisture', 'ndvi']
for col in FEATURE_COLS:
    print(f"  {col:20s}  min={df[col].min():.2f}  max={df[col].max():.2f}  mean={df[col].mean():.2f}")

print("""
⚠️  IMPORTANT: soil_moisture is in ABSOLUTE units (0–25), NOT percentage (0–100).
    UI slider sends 0–100%. Flask must convert: soil_model = soil_ui / 100 * 25
""")

# ═══════════════════════════════════════════════
# 2. FEATURES & SPLIT
# ═══════════════════════════════════════════════
X = df[FEATURE_COLS]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ═══════════════════════════════════════════════
# 3. MODEL with Calibration  (FIX A)
# ═══════════════════════════════════════════════
# CalibratedClassifierCV wraps the pipeline so predict_proba() gives
# reliable probabilities, not just rank scores.
# method='isotonic' works well for GBM; requires cv≥3.
base_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', GradientBoostingClassifier(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.05,
        min_samples_leaf=12,
        subsample=0.8,
        random_state=42
    ))
])

# FIX A: Calibrate on held-out folds → probabilities become meaningful
model = CalibratedClassifierCV(base_pipeline, cv=5, method='isotonic')
model.fit(X_train, y_train)

# ═══════════════════════════════════════════════
# 4. EVALUATE
# ═══════════════════════════════════════════════
y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("=== Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("=== ROC-AUC: ", round(roc_auc_score(y_test, y_proba), 4))
print("\n=== Classification Report:\n", classification_report(y_test, y_pred))
print("=== Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

cv     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_auc = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')
print(f"\n=== 5-fold CV AUC: {cv_auc.mean():.3f} ± {cv_auc.std():.3f}")

# ═══════════════════════════════════════════════
# 5. SANITY CHECKS  (FIX C — correct soil units)
# ═══════════════════════════════════════════════
THRESHOLD_HIGH   = 0.70   # FIX D: raised from 0.65
THRESHOLD_MEDIUM = 0.40

def classify(prob):
    if prob >= THRESHOLD_HIGH:   return "HIGH"
    if prob >= THRESHOLD_MEDIUM: return "MEDIUM"
    return "LOW"

def predict_case(rainfall, slope, elevation, soil_moisture=15.0, ndvi=0.45):
    """soil_moisture in MODEL units (0–25), NOT percentage."""
    df_in = pd.DataFrame(
        [[rainfall, slope, elevation, soil_moisture, ndvi]],
        columns=FEATURE_COLS
    )
    prob = model.predict_proba(df_in)[0][1]
    return prob, classify(prob)

print("\n=== Sanity checks (soil in model units 0–25, threshold HIGH=0.70) ===")
tests = [
    # label, rain, slope, elev, expected_levels
    ("DRY + gentle slope → LOW",          0,   10, 1800, ["LOW"]),
    ("DRY + steep slope  → LOW/MEDIUM",   0,   40, 2200, ["LOW", "MEDIUM"]),
    ("Heavy rain, flat   → LOW/MEDIUM", 150,    5, 1500, ["LOW", "MEDIUM"]),
    ("Shimla, heavy+steep → HIGH",        80,  45, 2206, ["HIGH"]),
    ("Manali, heavy+steep → HIGH",        90,  40, 2050, ["HIGH"]),
    ("Kullu,  heavy+steep → HIGH",        70,  35, 1220, ["HIGH", "MEDIUM"]),
    ("Shimla, dry+gentle → LOW",           3,   9, 2206, ["LOW"]),
    ("Mandi,  dry+gentle → LOW",           5,  10,  850, ["LOW"]),
]

all_passed = True
for label, rain, slp, elev, expected in tests:
    prob, risk = predict_case(rain, slp, elev)
    passed = risk in expected
    if not passed: all_passed = False
    status = "✅" if passed else "❌"
    print(f"  {status} {label}")
    print(f"     rain={rain}mm  slope={slp}°  elev={elev}m → prob={prob:.3f}  {risk}\n")

# ═══════════════════════════════════════════════
# 6. SAVE
# ═══════════════════════════════════════════════
if all_passed:
    joblib.dump(model, "C:/Users/babur/OneDrive/Desktop/ML_floodPrediction/model/landslide_model_v6.pkl")
    # FIX E: Save metadata so Flask can document exact unit expectations
    metadata = {
        "feature_cols":       FEATURE_COLS,
        "soil_moisture_unit": "absolute (0–25), NOT percentage",
        "soil_conversion":    "soil_model = soil_ui_percent / 100 * 25",
        "threshold_high":     THRESHOLD_HIGH,
        "threshold_medium":   THRESHOLD_MEDIUM,
        "elevation_source":   "district lookup table in Flask, NOT user input",
        "ndvi_default":       0.45,
    }
    joblib.dump(metadata, "C:/Users/babur/OneDrive/Desktop/ML_floodPrediction/model/landslide_model_v6_metadata.pkl")
    print("✅ All checks passed — model saved as landslide_model_v6.pkl")
else:
    print("❌ Some sanity checks failed — model NOT saved. Review above.")

=== Dataset shape: (570, 8)

Label counts:
label
0    475
1     95
Name: count, dtype: int64

Class balance: 16.7% positive

=== Correlation with label:
label            1.000
rainfall_7day    0.264
slope            0.229
soil_moisture    0.166
ndvi             0.097
aspect           0.082
elevation       -0.132
Name: label, dtype: float64

✅ rainfall_7day correlation = 0.264


Feature ranges (DOCUMENT THESE — Flask must match units!):
  rainfall_7day         min=0.00  max=396.04  mean=36.36
  slope                 min=0.00  max=67.22  mean=18.87
  elevation             min=320.00  max=5904.00  mean=2254.30
  soil_moisture         min=1.82  max=25.40  mean=17.04
  ndvi                  min=-1.00  max=1.00  mean=0.35

⚠️  IMPORTANT: soil_moisture is in ABSOLUTE units (0–25), NOT percentage (0–100).
    UI slider sends 0–100%. Flask must convert: soil_model = soil_ui / 100 * 25

=== Accuracy: 0.9035
=== ROC-AUC:  0.8698

=== Classification Report:
               precision    recall  f1-s